In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Pinecone  # Changed from FAISS to Pinecone
from pinecone import Pinecone, ServerlessSpec
from dotenv import load_dotenv
import os
from openai import OpenAI
import json

In [3]:
from docx import Document

def extract_text(file_path):
    doc = Document(file_path)
    full_text = []
    
    for para in doc.paragraphs:
        full_text.append(para.text)
    
    return "\n".join(full_text)

text1 = extract_text(r"indurance_docs\\The_IRDA_Act_1999_Attachment-1.docx")
text2 = extract_text(r"indurance_docs\\1938 _The_Insurance_Act_1938.docx")

print(text1[:500])

INSURANCE REGULATORY AND DEVELOPMENT AUTHORITY ACT, 1999
An Act
To provide for the establishment of an Authority to protect the interests of holders of insurance policies, to regulate, promote and ensure orderly growth of the insurance industry and for matters connected therewith or incidental thereto and further to amdend the Insurance Act, 1938, the Life Insurance Corporation Act, 1956 and the General Insurance Business(Nationalisation) Act, 1972.
BE it enacted by Parliament in Fiftieth Year o


In [3]:
load_dotenv()

True

In [4]:

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [5]:
index_name = os.getenv("PINECONE_INDEX_NAME")

In [7]:
embedding_tech = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [29]:
records = []

In [30]:
with open(r"indurance_docs\the_irda_act_1999_attachment.json", "r") as f:
    chunks = json.load(f)

In [12]:

def embed_text(text):
    embedding_ = embedding_tech.embed_query(text)
    return embedding_

In [31]:
for chunk in chunks:
    embedding = embed_text(chunk["text"])

    records.append({
        "id": chunk["chunk_id"],
        "values": embedding,
        "metadata": {
            "doc_id": chunk["doc_id"],
            "section_id": chunk["section_id"],
            "subsection_id": chunk["subsection_id"],
            "clause_id": chunk["clause_id"],
            "chunk_type": chunk["chunk_type"],
            "text": chunk["text"]
        }
    })

In [32]:
# Upsert in batches
batch_size = 100
index = pc.Index(index_name)
for i in range(0, len(records), batch_size):
    index.upsert(vectors=records[i:i+batch_size])

In [33]:
query = "What are the powers of IRDA authority?"

query_vector = embed_text(query)

results = index.query(
    vector=query_vector,
    top_k=5,
    include_metadata=True
)

In [34]:
results

{'matches': [{'id': '7232cea2-1bd2-400a-85c9-ed07c49a8222',
              'metadata': {'chunk_type': 'subsection',
                           'clause_id': 'unknown',
                           'doc_id': 'REGULATORY_ACT',
                           'section_id': '23',
                           'subsection_id': '2',
                           'text': 'The Authority may, by a general or special '
                                   'order in writing, also form committees of '
                                   'the members and delegate to them the '
                                   'powers and functions of the Authority as '
                                   'may be specified by the regulations.'},
              'score': 0.575363219,
              'values': []},
             {'id': 'caa59af5-2b29-49a5-9843-8ef3603399a6',
              'metadata': {'chunk_type': 'section',
                           'clause_id': 'unknown',
                           'doc_id': 'INSURANCE_ACT_1938',
     

search

In [35]:
results = index.query(
    vector=query_vector,
    top_k=5,
    include_metadata=True,
    filter={
        "doc_id": "REGULATORY_ACT"
    }
)

In [36]:
results

{'matches': [{'id': '7232cea2-1bd2-400a-85c9-ed07c49a8222',
              'metadata': {'chunk_type': 'subsection',
                           'clause_id': 'unknown',
                           'doc_id': 'REGULATORY_ACT',
                           'section_id': '23',
                           'subsection_id': '2',
                           'text': 'The Authority may, by a general or special '
                                   'order in writing, also form committees of '
                                   'the members and delegate to them the '
                                   'powers and functions of the Authority as '
                                   'may be specified by the regulations.'},
              'score': 0.575363219,
              'values': []},
             {'id': 'fc97b020-9818-48ad-ad47-3bc73fde01b1',
              'metadata': {'chunk_type': 'clause',
                           'clause_id': 'c',
                           'doc_id': 'REGULATORY_ACT',
                